# Validation 3: English → Hindi translation (Sarvam vs Google)

1. Run cells **2 → 5** once (setup + providers + batch runner + Gradio helpers)
2. Run cell **7** for full comparison across all headlines, or pick a headline section below

Output: Gradio panel with **English input** (left) and **Hindi translation** (right), plus latency and cost.<br><br>
Providers compared:<br>
- **Sarvam Mayura v1** — formal and colloquial modes (~1,000 char limit)<br>
- **Sarvam-Translate v1** — longer text (~2,000 char limit)<br>
- **Google Translate** — baseline via deep-translator<br><br>
Headline images live in `test_inputs/images to test translation/`; text corpus is defined in the notebook below.

## Summary of results

| Sr no | Headline | Provider | What it tests? | Hindi output quality | Latency | Cost (INR) | Comments |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | h01 coal miners | sarvam_mayura_formal | Formal policy tone | Pass | 688ms | 0.36 | Best for publishable Hindi headlines |
| 2 | h01 coal miners | sarvam_mayura_colloquial | Colloquial mode | Fail (Hinglish) | 473ms | 0.36 | Not colloquial Hindi |
| 3 | h01 coal miners | sarvam_translate_v1 | Long-context translate | Pass | 377ms | 0.54 | Fastest Sarvam on h01 |
| 4 | h01 coal miners | google_translate | Baseline | Pass | 1098ms | 0.31 | ~3x slower than Sarvam-translate |
| 5 | h02 wind energy | sarvam_mayura_formal | Energy headline | Pass | 494ms | 0.32 | Consistent formal tone |
| 6 | h02 wind energy | sarvam_mayura_colloquial | Colloquial mode | Fail (Hinglish) | 448ms | 0.32 | Not for Hindi-only output |
| 7 | h02 wind energy | sarvam_translate_v1 | Long-context translate | Pass | 363ms | 0.48 | Fastest in batch (~375ms avg) |
| 8 | h02 wind energy | google_translate | Baseline | Pass | 1076ms | 0.27 | Slowest on h02 |
| 9 | h03 AQI enforcement | sarvam_mayura_formal | Technical policy tone | Pass | 459ms | 0.38 | Good domain terms |
| 10 | h03 AQI enforcement | sarvam_mayura_colloquial | Colloquial mode | Fail (Hinglish) | 409ms | 0.38 | Same colloquial issue |
| 11 | h03 AQI enforcement | sarvam_translate_v1 | Long-context translate | Partial (AQI in Latin) | 385ms | 0.57 | OK if acronyms kept |
| 12 | h03 AQI enforcement | google_translate | Baseline | Partial (AQI in Latin) | 662ms | 0.32 | Similar to Sarvam-translate |
| — | **Overall** | — | — | Formal + Translate v1 reliable; colloquial not | Sarvam-translate ~375ms; Google ~945ms | Mayura ~0.35; Translate ~0.53; Google ~0.30 | Use formal or sarvam-translate:v1 for production Hindi |

## Setup — paths, API key, test corpus

In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
import gradio as gr
from deep_translator import GoogleTranslator
from sarvamai import SarvamAI

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

# headline screenshots (for visual reference during manual review)
IMAGE_DIR = ROOT / "test_inputs/images to test translation"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY", "")
USD_TO_INR = 85  # rough conversion for Google cost estimate only

_sarvam_client: SarvamAI | None = None


def get_sarvam_client() -> SarvamAI:
    """Reuse one Sarvam client across provider calls."""
    global _sarvam_client
    if _sarvam_client is None:
        if not SARVAM_API_KEY:
            raise ValueError("SARVAM_API_KEY not set — add it to .env")
        _sarvam_client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
    return _sarvam_client


# test corpus — three English headlines (title + subtitle joined, same text we OCR'd from images)
HEADLINES = {
    "h01": {
        "title": "An Inclusive Transition Must Recognise Coal Miners in India's Green Future",
        "subtitle": "Coal miners' pivotal role in India's economy and energy sector calls for inclusion in greener transitions",
        "image": "headline1.png",
    },
    "h02": {
        "title": "Harnessing the Wind for India's Ambitious Energy Transition",
        "subtitle": "India ranks fourth globally in total wind installations, but it's time to step on the accelerator.",
        "image": "headline2.png",
    },
    "h03": {
        "title": "Why AQI alone cannot guide effective air quality enforcement in India",
        "subtitle": "Tackling emission sources along with AQI reduction would lead to sustained air quality improvement throughout the year.",
        "image": "headline3.png",
    },
}


def headline_text(hid: str) -> str:
    """Build the single string we send to every translation provider."""
    h = HEADLINES[hid]
    return f"{h['title']}. {h['subtitle']}"


print("ROOT:", ROOT)
print("SARVAM_API_KEY:", "set" if SARVAM_API_KEY else "MISSING")
print("Headline images:", sorted(p.name for p in IMAGE_DIR.glob("*.png")))
print("Headline ids:", list(HEADLINES.keys()))

/Users/mirambikasikdar/Desktop/Personal/Prep/Coding/comparison-of-LLM-models/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: /Users/mirambikasikdar/Desktop/Personal/Prep/Coding/comparison-of-LLM-models
SARVAM_API_KEY: set
Headline images: ['headline1.png', 'headline2.png', 'headline3.png']
Headline ids: ['h01', 'h02', 'h03']


## Translation providers

Each function takes English text and returns the same dict shape so comparison stays fair:<br>
`output`, `latency_ms`, `input_chars`, `output_chars`, `model`, `provider`, `extra`

In [4]:
def sarvam_mayura(
    text: str,
    source_lang: str = "en-IN",
    target_lang: str = "hi-IN",
    mode: str = "formal",
) -> dict:
    """Sarvam Mayura v1 — best for shorter headlines; mode switches formal vs colloquial tone."""
    client = get_sarvam_client()
    input_chars = len(text)

    start = time.perf_counter()
    resp = client.text.translate(
        input=text,
        source_language_code=source_lang,
        target_language_code=target_lang,
        model="mayura:v1",
        mode=mode,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    translated = resp.translated_text if hasattr(resp, "translated_text") else str(resp)

    return {
        "output": translated,
        "latency_ms": round(latency_ms, 2),
        "input_chars": input_chars,
        "output_chars": len(translated),
        "model": "mayura:v1",
        "provider": "sarvam",
        "extra": {"mode": mode, "max_input_chars": 1000},
    }


def sarvam_translate_v1(
    text: str,
    source_lang: str = "en-IN",
    target_lang: str = "hi-IN",
) -> dict:
    """Sarvam-Translate v1 — handles slightly longer input (~2k chars); no mode param."""
    client = get_sarvam_client()
    input_chars = len(text)

    start = time.perf_counter()
    resp = client.text.translate(
        input=text,
        source_language_code=source_lang,
        target_language_code=target_lang,
        model="sarvam-translate:v1",
    )
    latency_ms = (time.perf_counter() - start) * 1000
    translated = resp.translated_text if hasattr(resp, "translated_text") else str(resp)

    return {
        "output": translated,
        "latency_ms": round(latency_ms, 2),
        "input_chars": input_chars,
        "output_chars": len(translated),
        "model": "sarvam-translate:v1",
        "provider": "sarvam",
        "extra": {"mode": "formal", "max_input_chars": 2000},
    }


def google_translate(
    text: str,
    source_lang: str = "en",
    target_lang: str = "hi",
) -> dict:
    """Google Translate baseline — useful sanity check, not tuned for Indic nuance."""
    input_chars = len(text)

    start = time.perf_counter()
    translated = GoogleTranslator(source=source_lang, target=target_lang).translate(text) or ""
    latency_ms = (time.perf_counter() - start) * 1000

    return {
        "output": translated,
        "latency_ms": round(latency_ms, 2),
        "input_chars": input_chars,
        "output_chars": len(translated),
        "model": "google-translate",
        "provider": "google",
        "extra": {},
    }


# registry — keys match what we log in the summary table
TRANSLATE_PROVIDERS = {
    "sarvam_mayura_formal": lambda t: sarvam_mayura(t, mode="formal"),
    "sarvam_mayura_colloquial": lambda t: sarvam_mayura(t, mode="modern-colloquial"),
    "sarvam_translate_v1": sarvam_translate_v1,
    "google_translate": google_translate,
}

print("Providers:", list(TRANSLATE_PROVIDERS.keys()))

Providers: ['sarvam_mayura_formal', 'sarvam_mayura_colloquial', 'sarvam_translate_v1', 'google_translate']


## Cost estimate + batch comparison runner

In [5]:
def estimate_translation_cost_inr(provider: str, model: str, input_chars: int) -> float:
    """Rough INR estimate per call — Sarvam list prices, Google converted via USD_TO_INR."""
    if provider == "sarvam" and model == "mayura:v1":
        return round(input_chars / 10_000 * 20, 4)
    if provider == "sarvam" and model == "sarvam-translate:v1":
        return round(input_chars / 10_000 * 30, 4)
    if provider == "google":
        cost_usd = input_chars / 1_000_000 * 20
        return round(cost_usd * USD_TO_INR, 4)
    return 0.0


def run_translation_comparison() -> list[dict]:
    """Run every headline through every provider; print per-call logs + summary table."""
    results: list[dict] = []

    for hid, headline in HEADLINES.items():
        full_text = headline_text(hid)

        print(f"\n{'=' * 70}")
        print(f"[{hid}] {headline['title'][:60]}...")
        print(f"{'=' * 70}")

        for provider_name, provider_fn in TRANSLATE_PROVIDERS.items():
            print(f"\n  → {provider_name}...", end=" ", flush=True)

            try:
                result = provider_fn(full_text)
            except Exception as e:
                print(f"ERROR: {e}")
                results.append({
                    "headline_id": hid,
                    "provider": provider_name,
                    "error": str(e),
                })
                continue

            cost = estimate_translation_cost_inr(
                result["provider"], result["model"], result["input_chars"]
            )

            print(f"{result['latency_ms']:.0f}ms | ₹{cost}")
            print(f"    {result['output'][:100]}...")

            results.append({
                "headline_id": hid,
                "provider": provider_name,
                "model": result["model"],
                "english": full_text,
                "hindi": result["output"],
                "latency_ms": result["latency_ms"],
                "input_chars": result["input_chars"],
                "output_chars": result["output_chars"],
                "cost_inr": cost,
            })

    print(f"\n\n{'=' * 70}")
    print("TRANSLATION COMPARISON SUMMARY")
    print(f"{'=' * 70}")
    print(f"{'Provider':<30} {'Avg Latency':>12} {'Avg Cost':>10} {'Avg Out Chars':>14}")
    print("-" * 70)

    for pname in TRANSLATE_PROVIDERS:
        p_results = [r for r in results if r.get("provider") == pname and "error" not in r]
        if p_results:
            avg_lat = sum(r["latency_ms"] for r in p_results) / len(p_results)
            avg_cost = sum(r["cost_inr"] for r in p_results) / len(p_results)
            avg_chars = sum(r["output_chars"] for r in p_results) / len(p_results)
            print(f"{pname:<30} {avg_lat:>10.0f}ms {avg_cost:>9.4f}₹ {avg_chars:>12.0f}")

    return results

## Gradio viewer + single-headline helper

In [ ]:
def show_translation_result(english: str, result: dict, *, window_height: int = 500) -> gr.Blocks:
    """Side-by-side view: English source | Hindi output + latency/cost line."""
    cost = estimate_translation_cost_inr(result["provider"], result["model"], result["input_chars"])
    status = (
        f"**{result['provider']}** / `{result['model']}` · "
        f"{result['latency_ms']:.0f}ms · ₹{cost} · "
        f"{result['input_chars']} → {result['output_chars']} chars"
    )
    extra = result.get("extra") or {}
    if extra:
        status += f" · extra: `{extra}`"

    with gr.Blocks() as demo:
        gr.Markdown(status)
        with gr.Row(equal_height=True):
            gr.Textbox(value=english, label="English (input)", lines=6, interactive=False)
            gr.Textbox(value=result["output"], label="Hindi (output)", lines=6, interactive=False)
    demo.launch(inline=True, height=window_height, share=False, quiet=True)
    return demo


def translate_headline(hid: str, provider_name: str) -> tuple[str, dict]:
    """Translate one headline id with a named provider from TRANSLATE_PROVIDERS."""
    if hid not in HEADLINES:
        raise ValueError(f"Unknown headline {hid}. Choose from {list(HEADLINES)}")
    if provider_name not in TRANSLATE_PROVIDERS:
        raise ValueError(f"Unknown provider. Choose from {list(TRANSLATE_PROVIDERS)}")
    english = headline_text(hid)
    result = TRANSLATE_PROVIDERS[provider_name](english)
    return english, result

## Run all headlines × all providers

In [6]:
comparison_results = run_translation_comparison()


[h01] An Inclusive Transition Must Recognise Coal Miners in India'...

  → sarvam_mayura_formal... 688ms | ₹0.362
    एक समावेशी संक्रमण को भारत के हरित भविष्य में कोयला खनिकों को मान्यता देनी चाहिए। भारत के अर्थव्यवस्...

  → sarvam_mayura_colloquial... 473ms | ₹0.362
    एक inclusive transition को India के green future में coal miners को recognize करना चाहिए। India की e...

  → sarvam_translate_v1... 377ms | ₹0.543
    एक समावेशी परिवर्तन को भारत के हरित भविष्य में कोयला खनिकों को मान्यता देनी चाहिए। भारत की अर्थव्यवस...

  → google_translate... 1098ms | ₹0.3077
    एक समावेशी परिवर्तन को भारत के हरित भविष्य में कोयला खनिकों को अवश्य पहचानना चाहिए। भारत की अर्थव्यव...

[h02] Harnessing the Wind for India's Ambitious Energy Transition...

  → sarvam_mayura_formal... 494ms | ₹0.318
    भारत के महत्वाकांक्षी ऊर्जा संक्रमण के लिए पवन ऊर्जा का उपयोग। भारत कुल पवन प्रतिष्ठानों में विश्व स...

  → sarvam_mayura_colloquial... 448ms | ₹0.318
    India के ambitious energy transition के लिए wi

## Quick smoke test — Sarvam Mayura (formal)

Short string to confirm API key and provider wiring before running the full batch.

In [9]:
sample = "An Inclusive Transition Must Recognise Coal Miners"
mayura_result = sarvam_mayura(sample, mode="formal")
mayura_result

{'output': 'एक समावेशी संक्रमण को कोयला खनिकों को मान्यता देनी चाहिए।',
 'latency_ms': 384.07,
 'input_chars': 50,
 'output_chars': 57,
 'model': 'mayura:v1',
 'provider': 'sarvam',
 'extra': {'mode': 'formal', 'max_input_chars': 1000}}